## Imports

In [15]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
from assignment10_functions import *


## **Mediapipe vs Ground truth (measured lengths) And Kinect** 


### Ground truth distances between joind (Enis)

In [16]:
ground_truth = {
    "hip_to_shoulder":      55,
    "knee_to_hip":          47,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}

### 2D Joint Distance Estimation from a Static Video


In [17]:
df = pd.read_csv("../../EnisProject/data/still_video_mediapipe.csv")

REFERENCE = "hip_to_shoulder"


df = convert_to_pixel_coordinates(df, 1920, 1080)
df_dist = add_2D_distances(df)

avg = df_dist[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()


scale = ground_truth[REFERENCE] / avg[REFERENCE]
print(f"Scale factor: {scale:.2f} cm per unit\n")

print_comparison_table(
    avg=avg,
    ground_truth=ground_truth,
    scale=scale,
    reference=REFERENCE)

Scale factor: 0.18 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 55.0            55.0          0.0        0.0%
knee_to_hip                     47.0            46.1          0.9        1.9%
hip_to_hip                      25.0            21.6          3.4       13.6%
knee_to_ankle                   43.0            38.9          4.1        9.5%
shoulder_to_shoulder            38.0            34.5          3.5        9.3%
---------------------------------------------------------------------------
Average error                                                 3.0


### 3D Joint Distance Estimation from a Static Video


In [18]:
df = pd.read_csv("../../EnisProject/data/still_video_world_mediapipe.csv")

REFERENCE = "hip_to_shoulder"


df_dist = add_3D_distances(df)

avg = df_dist[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()


scale = ground_truth[REFERENCE] / avg[REFERENCE]
print(f"Scale factor: {scale:.2f} cm per unit\n")

print_comparison_table(
    avg=avg,
    ground_truth=ground_truth,
    scale=scale,
    reference=REFERENCE)

Scale factor: 112.11 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 55.0            55.0          0.0        0.0%
knee_to_hip                     47.0            46.3          0.7        1.5%
hip_to_hip                      25.0            25.7          0.7        2.7%
knee_to_ankle                   43.0            42.6          0.4        0.9%
shoulder_to_shoulder            38.0            36.1          1.9        4.9%
---------------------------------------------------------------------------
Average error                                                 0.9


### 3D Joint Distance Estimation from a Dynamic Video


In [19]:
df = pd.read_csv("../../EnisProject/data/1m_step_high_world_mediapipe.csv")

REFERENCE = "hip_to_shoulder"

df_dist = add_3D_distances(df)

avg = df_dist[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()


scale = ground_truth[REFERENCE] / avg[REFERENCE]
print(f"Scale factor: {scale:.2f} cm per unit\n")

print_comparison_table(
    avg=avg,
    ground_truth=ground_truth,
    scale=scale,
    reference=REFERENCE)

Scale factor: 113.81 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 55.0            55.0          0.0        0.0%
knee_to_hip                     47.0            47.9          0.9        1.9%
hip_to_hip                      25.0            25.5          0.5        2.0%
knee_to_ankle                   43.0            41.9          1.1        2.7%
shoulder_to_shoulder            38.0            36.6          1.4        3.7%
---------------------------------------------------------------------------
Average error                                                 1.0


### 3D Joint Distance Comparison Between Kinect and MediaPipe from a Dynamic Video


In [20]:
kinect_df = pd.read_csv("..//Assignment9/data/kinect_good_preprocessed/A1_kinect.csv")
mp_df = pd.read_csv("../../EnisProject/data/A1_world_mediapipe.csv")

REFERENCE = "hip_to_shoulder"



mp_aligned, kinect_ = align_by_frame(mp_df, kinect_df)

kinect_dist = add_3D_distances(kinect_)
mp_dist = add_3D_distances(mp_aligned)


scale = kinect_dist[REFERENCE].mean() / mp_dist[REFERENCE].mean()
mp_scaled = mp_dist * scale


errors = abs(mp_scaled - kinect_dist)
print(f"Scale factor: {scale:.4f}\n")


print_kinect_mp_comparison(
    kinect_dist=kinect_dist,
    mp_scaled=mp_scaled,
    errors=errors,)

Scale factor: 0.9597

Measurement                     Kinect    MediaPipe        Error    Error %
---------------------------------------------------------------------------
hip_to_shoulder                   50.1         50.1          1.0        1.9%
knee_to_hip                       36.0         37.0          1.6        4.5%
hip_to_hip                        14.7         21.5          6.9       47.0%
knee_to_ankle                     37.8         33.5          4.3       11.4%
shoulder_to_shoulder              29.4         28.7          1.5        5.2%
---------------------------------------------------------------------------
Average error                                                3.1
